In [1]:
# Imports
import json
import os
import pickle
import shutil
import subprocess
import sys
import time
from pathlib import Path

import requests


In [17]:
# Setting the project folder
project_dir = Path(r"C:\Users\HomePC\ML Zoomcamp 2026\Homework\Homework 5 Deploying Machine Learning Models")

# The Dockerfile needs this file.
python_version_file = project_dir / ".python-version"
if not python_version_file.exists():
    python_version_file.write_text("3.11.15\n", encoding="utf-8")

## Question 1 — environment check

Run `uv --version` and record the version in your notes. This is a setup check, not a graded answer: your local `uv` version is not a property of the model.


In [19]:
result = subprocess.run(["uv", "--version"], cwd=project_dir, capture_output=True, text=True)

print(result.stdout.strip() or result.stderr.strip())

uv 0.13.1 (62faed335 2026-10-13 x86_64-pc-windows-msvc)


## Question 2 — locked dependency

Open `pyproject.toml` and `uv.lock`. Which Scikit-Learn version is part of the 2026 reference environment?

- `1.6.1`
- `1.7.2`
- `1.8.0`
- `2.0.0`

Do not regenerate the lockfile for the graded run; use `uv sync --locked`.


In [23]:
# Installing the versions recorded in uv.lock.
result = subprocess.run(["uv", "sync", "--locked"], cwd=project_dir, capture_output=True, text=True)

print(result.stdout)
print(result.stderr)


Using CPython 3.11.15
Creating virtual environment at: .venv
Resolved 19 packages in 1ms
Installed 18 packages in 1.06s
 + annotated-types==0.8.0
 + anyio==4.15.1
 + click==8.5.0
 + cloudpickle==3.1.2
 + fastapi==0.119.1
 + h11==0.16.0
 + idna==3.19
 + joblib==1.6.0
 + numpy==2.3.3
 + pydantic==2.13.5
 + pydantic-core==2.46.5
 + scikit-learn==1.7.2
 + scipy==1.17.1
 + starlette==0.48.0
 + threadpoolctl==3.6.0
 + typing-extensions==4.16.0
 + typing-inspection==0.4.4
 + uvicorn==0.38.0



## Question 3 — load the model

Use `pickle` to load `pipeline.bin` and call `predict_proba` for this lead:

```python
{
    "lead_source": "paid_ads",
    "industry": "technology",
    "employment_status": "employed",
    "location": "north_america",
    "number_of_courses_viewed": 2,
    "annual_income": 79276.0,
    "interaction_count": 4,
    "lead_score": 0.41,
}
```

Report the probability rounded to three decimal places.

In [25]:
with open(project_dir / "pipeline.bin", "rb") as f:
    pipeline = pickle.load(f)

lead_q3 = {
    "lead_source": "paid_ads",
    "industry": "technology",
    "employment_status": "employed",
    "location": "north_america",
    "number_of_courses_viewed": 2,
    "annual_income": 79276.0,
    "interaction_count": 4,
    "lead_score": 0.41,
}

probability = pipeline.predict_proba([lead_q3])[0, 1]
print(round(float(probability), 3))

0.533


## Question 4 — serve the model

Start the API with:

```bash
uv run uvicorn predict:app --host 0.0.0.0 --port 9696
```

Send this lead to `POST /predict`:

```python
{
    "lead_source": "organic_search",
    "industry": "technology",
    "employment_status": "employed",
    "location": "europe",
    "number_of_courses_viewed": 4,
    "annual_income": 80304.0,
    "interaction_count": 7,
    "lead_score": 0.74,
}
```

In [29]:
# Starting the API in the background if it is not already running.
api_process = globals().get("api_process")
if api_process is None or api_process.poll() is not None:
    api_process = subprocess.Popen([sys.executable, "-m", "uvicorn", "predict:app","--host", "127.0.0.1", "--port", "9696",],
        cwd=project_dir,
        stdout=subprocess.DEVNULL,
        stderr=subprocess.DEVNULL )

api_url = "http://127.0.0.1:9696"
time.sleep(2)

print(requests.get(api_url + "/health").json())

lead_q4 = {
    "lead_source": "organic_search",
    "industry": "technology",
    "employment_status": "employed",
    "location": "europe",
    "number_of_courses_viewed": 4,
    "annual_income": 80304.0,
    "interaction_count": 7,
    "lead_score": 0.74,
}

response = requests.post(api_url + "/predict", json=lead_q4)
print(response.json())

{'status': 'ok', 'model_sha256': '1646bbdcd38d4f044da6b630c5b332c93a314245a8c21929011c42de51f629f1', 'feature_defaults_sha256': '1694f842b908fac9f53c0cc19be6d91f934befc63af23cc32fefafde69acd0cd'}
{'conversion_probability': 0.769799, 'conversion': True}


## Container

The Dockerfile uses Python `3.11.15-slim-bookworm`, `uv 0.10.11`, and the checked-in `uv.lock`.

Build and run the container:

```bash
docker build -t zoomcamp-model:2026-hw5 .
docker run --rm -p 9696:9696 zoomcamp-model:2026-hw5
```


## Question 5 — look at the container configuration

Which Python base-image tag is declared in the Dockerfile?

- `python:3.9-slim-bullseye`
- `python:3.11.15-slim-bookworm`
- `python:3.12-slim-bookworm`
- `python:3.13.10-slim-bookworm`


In [30]:
print((project_dir / "Dockerfile").read_text(encoding="utf-8"))

FROM python:3.11.15-slim-bookworm@sha256:d29f48a31a8b408ed19272ca1e7b10ebae13b240a27e862d3d4217c528e2e0c3

COPY --from=ghcr.io/astral-sh/uv:0.10.11@sha256:3472e43b4e738cf911c99d41bb34331280efad54c73b1def654a6227bb59b2b4 /uv /uvx /bin/

ENV PATH="/code/.venv/bin:$PATH" \
    UV_COMPILE_BYTECODE=1 \
    UV_LINK_MODE=copy

WORKDIR /code
COPY pyproject.toml uv.lock .python-version ./
RUN uv sync --locked --no-dev

COPY model.py predict.py feature_defaults.json model_metadata.json pipeline.bin q6_test.py ./

EXPOSE 9696
ENTRYPOINT ["uvicorn", "predict:app", "--host", "0.0.0.0", "--port", "9696"]



In [31]:
# Stopping the local API before using port 9696 for Docker.
if "api_process" in globals() and api_process is not None and api_process.poll() is None:
    api_process.terminate()
    api_process.wait(timeout=5)

print("Local API stopped.")

Local API stopped.


In [32]:
# Building the image.
result = subprocess.run(["docker", "build", "-t", "zoomcamp-model:2026-hw5", "."], cwd=project_dir, capture_output=True, text=True, timeout=600 )

print("Return code:", result.returncode)
print(result.stdout[-3000:])
print(result.stderr[-3000:])

Return code: 0

#0 building with "desktop-linux" instance using docker driver

#1 [internal] load build definition from Dockerfile
#1 transferring dockerfile: 637B done
#1 DONE 0.0s

#2 [internal] load metadata for docker.io/library/python:3.11.15-slim-bookworm@sha256:d29f48a31a8b408ed19272ca1e7b10ebae13b240a27e862d3d4217c528e2e0c3
#2 ...

#3 [internal] load metadata for ghcr.io/astral-sh/uv:0.10.11@sha256:3472e43b4e738cf911c99d41bb34331280efad54c73b1def654a6227bb59b2b4
#3 DONE 0.3s

#2 [internal] load metadata for docker.io/library/python:3.11.15-slim-bookworm@sha256:d29f48a31a8b408ed19272ca1e7b10ebae13b240a27e862d3d4217c528e2e0c3
#2 DONE 0.3s

#4 [internal] load .dockerignore
#4 transferring context: 2B done
#4 DONE 0.0s

#5 [internal] load build context
#5 transferring context: 297B done
#5 DONE 0.0s

#6 FROM ghcr.io/astral-sh/uv:0.10.11@sha256:3472e43b4e738cf911c99d41bb34331280efad54c73b1def654a6227bb59b2b4
#6 resolve ghcr.io/astral-sh/uv:0.10.11@sha256:3472e43b4e738cf911c99d41bb34

In [33]:
# Starting the container.
subprocess.run(["docker", "rm", "-f", "zoomcamp-model-hw5"], capture_output=True, text=True)

result = subprocess.run(
    [
        "docker", "run", "-d", "--rm",
        "--name", "zoomcamp-model-hw5",
        "-p", "9696:9696",
        "zoomcamp-model:2026-hw5",
    ],
    capture_output=True,
    text=True)

print("Return code:", result.returncode)
print("Container ID:", result.stdout.strip())
print(result.stderr)

Return code: 0
Container ID: 157853f3143c6db05dd0a9120acdf5984478fc2140d32a5246e59a98f57ce8ea



## Question 6 — run the container

Run the same request from Question 4 against the container. The provided smoke client can be run with:

```bash
python q6_test.py
```

Report `conversion_probability` rounded to three decimal places. The expected result is the same reference inference as Question 4, with the same `0.005` absolute tolerance. Also verify that:

```bash
curl -s http://localhost:9696/health
```

returns `{"status":"ok", ...}` and the model checksum recorded in `model_metadata.json`.

In [35]:
health = requests.get("http://127.0.0.1:9696/health").json()
print(health)

metadata = (project_dir / "model_metadata.json").read_text(encoding="utf-8")
print("Checksum appears in metadata:", health["model_sha256"] in metadata)

result = subprocess.run([sys.executable, str(project_dir / "q6_test.py")], cwd=project_dir, capture_output=True, text=True, timeout=30)

print("Return code:", result.returncode)
print(result.stdout)
print(result.stderr)

{'status': 'ok', 'model_sha256': '1646bbdcd38d4f044da6b630c5b332c93a314245a8c21929011c42de51f629f1', 'feature_defaults_sha256': '1694f842b908fac9f53c0cc19be6d91f934befc63af23cc32fefafde69acd0cd'}
Checksum appears in metadata: True
Return code: 0
{'conversion_probability': 0.769799, 'conversion': True}


